In [1]:
import os
os.chdir('C:\\Users\\User\\anaconda3\\envs\\first_env')

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseries.utils.to_split import to_split
from timeseries.utils.multivariate_multi_step import multivariate_multi_step
from timeseries.utils.multivariate_single_step import multivariate_single_step
from timeseries.utils.univariate_multi_step import univariate_multi_step
from timeseries.utils.univariate_single_step import univariate_single_step
from timeseries.utils.CosineAnnealingLRS import CosineAnnealingLRS

#from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
#from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, Reshape, Dense, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
#from keras.callbacks import Callback

In [8]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [9]:
def create_rnn():
    input_data = Input(shape=(time_steps, num_features))
    rnn_layer1 = SimpleRNN(8, return_sequences=True)(input_data)
    rnn_layer2 = SimpleRNN(20)(rnn_layer1)
    x = Flatten()(rnn_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

In [11]:
def create_rnn():
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import SimpleRNN, Dense

    model = Sequential([
        SimpleRNN(50, activation='relu', input_shape=(24, 21)),
        Dense(24)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


In [12]:
model1 = create_rnn()
model1.summary()

c:\Users\User\anaconda3\envs\first_env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 50)             │         3,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │         1,224 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,824 (18.84 KB)

 Trainable params: 4,824 (18.84 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import tensorflow
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [15]:
checkpoints =  r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [16]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
#TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1]

In [17]:

# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_rnn()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [22]:
import os
path_dataset =r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9'
path_tr = os.path.join(path_dataset, 'AEP_train.csv')
df_tr = pd.read_csv(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\AEP_train.csv')
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
df_v = pd.read_csv(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\AEP_validation.csv')
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'AEP_test.csv')
df_te = pd.read_csv(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\AEP_test.csv')
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')
scaler      = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\User\anaconda3\envs\first_env\lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((84907, 21), (24259, 21), (12130, 21))

In [23]:
time_steps=24
num_features=21

In [24]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 2.4939639568328857 sec


In [25]:
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/5
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0829 - mae: 0.0829 - mape: 4410.0850
Epoch 1: val_loss improved from inf to 0.01253, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0001-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - loss: 0.0828 - mae: 0.0828 - mape: 4404.9697 - val_loss: 0.0125 - val_mae: 0.0125 - val_mape: 6.2187
Epoch 2/5
2648/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0130 - mae: 0.0130 - mape: 11.5691
Epoch 2: val_loss improved from 0.01253 to 0.00932, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0002-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 28s 11ms/step - loss: 0.0130 - mae: 0.0130 - mape: 11.7011 - val_loss: 0.0093 - val_mae: 0.0093 - val_mape: 4.5756
Epoch 3/5
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0104 - mae: 0.0104 - mape: 28.2081
Epoch 3: val_loss improved from 0.00932 to 0.00904, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0003-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 34s 8ms/step - loss: 0.0104 - mae: 0.0104 - mape: 28.2280 - val_loss: 0.0090 - val_mae: 0.0090 - val_mape: 4.3771
Epoch 4/5
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0095 - mae: 0.0095 - mape: 99.7966
Epoch 4: val_loss improved from 0.00904 to 0.00824, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0004-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - loss: 0.0095 - mae: 0.0095 - mape: 99.7985 - val_loss: 0.0082 - val_mae: 0.0082 - val_mape: 3.7451
Epoch 5/5
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0090 - mae: 0.0090 - mape: 20.1551
Epoch 5: val_loss did not improve from 0.00824
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.0090 - mae: 0.0090 - mape: 20.2078 - val_loss: 0.0084 - val_mae: 0.0084 - val_mape: 3.8245


In [26]:

model = load_model(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0002-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Mean Absolute Error (MAE): 147.61
Median Absolute Error (MedAE): 119.73
Mean Squared Error (MSE): 36550.28
Root Mean Squared Error (RMSE): 191.18
Mean Absolute Percentage Error (MAPE): 1.02 %
Median Absolute Percentage Error (MDAPE): 0.82 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 24)


In [27]:
checkpoints = r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\E1-cp-0053-loss0.01.h5'
start_epoch= 54

In [28]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading model from disk...")
    model = load_model(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0002-loss0.01.h5', compile=False)

    # Manually compile since we used compile=False
    opt = Adam(1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])

    # Print and update learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    model.optimizer.learning_rate = 1e-4

    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))


[INFO] loading model from disk...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [29]:
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/5
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0090 - mae: 0.0090 - mape: 17.0724
Epoch 1: val_loss did not improve from 0.00824
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0090 - mae: 0.0090 - mape: 17.1515 - val_loss: 0.0086 - val_mae: 0.0086 - val_mape: 4.1240
Epoch 2/5
2645/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0087 - mae: 0.0087 - mape: 58.8864
Epoch 2: val_loss did not improve from 0.00824
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.0087 - mae: 0.0087 - mape: 59.0628 - val_loss: 0.0084 - val_mae: 0.0084 - val_mape: 3.9079
Epoch 3/5
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0085 - mae: 0.0085 - mape: 12.5101
Epoch 3: val_loss improved from 0.00824 to 0.00822, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0003-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 42s 8ms/step - loss: 0.0085 - mae: 0.0085 - mape: 12.5517 - val_loss: 0.0082 - val_mae: 0.0082 - val_mape: 3.8502
Epoch 4/5
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0083 - mae: 0.0083 - mape: 51.2618
Epoch 4: val_loss improved from 0.00822 to 0.00810, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0004-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.0083 - mae: 0.0083 - mape: 51.3751 - val_loss: 0.0081 - val_mae: 0.0081 - val_mape: 3.9514
Epoch 5/5
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0081 - mae: 0.0081 - mape: 213.1592
Epoch 5: val_loss improved from 0.00810 to 0.00798, saving model to C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0005-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.0081 - mae: 0.0081 - mape: 212.9693 - val_loss: 0.0080 - val_mae: 0.0080 - val_mape: 3.7167


In [30]:

model = load_model(r'C:\\Users\\User\\anaconda3\\envs\\first_env\Lab_9\\E1-cp-0002-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Mean Absolute Error (MAE): 147.61
Median Absolute Error (MedAE): 119.73
Mean Squared Error (MSE): 36550.28
Root Mean Squared Error (RMSE): 191.18
Mean Absolute Percentage Error (MAPE): 1.02 %
Median Absolute Percentage Error (MDAPE): 0.82 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 24)
